# Train SigExt — Full 27-Model Matrix

Trains **27 SigExt models** across the complete experimental matrix:

| Language | Base Model | Sample Sizes | Thresholds |
|---|---|---|---|
| **English** | `allenai/longformer-base-4096` | 1k, 2.5k, 5k | 0.60, 0.70, 0.80 |
| **English** | `markussagen/xlm-roberta-longformer-base-4096` | 1k, 2.5k, 5k | 0.60, 0.70, 0.80 |
| **Italian** | `markussagen/xlm-roberta-longformer-base-4096` | 1k, 2.5k, 5k | 0.60, 0.70, 0.80 |

**Seed**: 42 (fixed for all). All models pushed to HuggingFace Hub.

**Checkpointing**: Progress is saved to `training_checkpoint.json`. If the notebook is interrupted, rerunning it will skip already completed models.

In [ ]:
import warnings, os, json
from dotenv import load_dotenv
warnings.filterwarnings('ignore')
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

!uv pip install -e ../..
load_dotenv()
from huggingface_hub import login
login(token=os.getenv('HF_TOKEN'))

: 

In [ ]:
from sm_sip.config import TrainingConfig, SigExtConfig
from sm_sip.pipelines.training import train_sigext
from sm_sip.utils.gpu import clear_gpu_memory

## Training Matrix (27 configs)

In [ ]:
CHECKPOINT_FILE = "training_checkpoint.json"

def load_checkpoint():
    if os.path.exists(CHECKPOINT_FILE):
        with open(CHECKPOINT_FILE, "r") as f:
            return json.load(f)
    return []

def save_checkpoint(completed_models):
    with open(CHECKPOINT_FILE, "w") as f:
        json.dump(completed_models, f, indent=4)

completed_models = load_checkpoint()
print(f"Found {len(completed_models)} completed models in checkpoint.")

In [ ]:
configs = []

for lang, ld in SigExtConfig.LANG_DATASETS.items():
    for base_key in SigExtConfig.LANG_BASE_MODELS[lang]:
        base_model_id = SigExtConfig.BASE_MODELS[base_key]
        for n in SigExtConfig.SAMPLE_SIZES:
            for t in SigExtConfig.THRESHOLDS:
                name_n = f'{n // 1000}k' if n >= 1000 and n % 1000 == 0 else str(n)
                t_str = f'{t:.2f}'.replace('.', '')
                output_name = f'sigext-{ld["prefix"]}-{lang}-{base_key}-{name_n}-{t_str}t'
                
                configs.append(TrainingConfig(
                    lang=lang,
                    base_model_id=base_model_id,
                    dataset_name=ld['dataset'],
                    num_samples=n,
                    similarity_threshold=t,
                    output_model_name=output_name,
                    push_to_hub=True,
                    seed=SigExtConfig.SEED,
                ))

print(f'Total training configs: {len(configs)}')
print(f'\n{"Model Name":<50} {"Base":>12} {"Lang":>4} {"N":>6} {"Thr":>5}')
print('-' * 80)
for c in configs:
    base_short = c.base_model_id.split('/')[-1][:12]
    status = "[DONE]" if c.output_model_name in completed_models else ""
    print(f'{c.output_model_name:<50} {base_short:>12} {c.lang:>4} {c.num_samples:>6} {c.similarity_threshold:>5.2f} {status}')

## Training Loop

In [ ]:
for i, config in enumerate(configs):
    if config.output_model_name in completed_models:
        print(f"Skip {config.output_model_name} (already completed)")
        continue

    print(f'\n{"="*70}')
    print(f'  [{i+1}/{len(configs)}] {config.output_model_name}')
    print(f'  Base: {config.base_model_id}')
    print(f'  Lang: {config.lang}, Samples: {config.num_samples}, Threshold: {config.similarity_threshold}')
    print(f'  Seed: {config.seed}')
    print(f'{"="*70}')

    try:
        model, tokenizer = train_sigext(config)
        del model, tokenizer
        clear_gpu_memory()
        
        # Update checkpoint
        completed_models.append(config.output_model_name)
        save_checkpoint(completed_models)
        
        print(f'  {config.output_model_name} complete!')
    except Exception as e:
        print(f'  FAILED: {e}')
        clear_gpu_memory()
        continue

print(f'\n\nAll {len(configs)} training runs complete!')